# Artigo 0008 - LLMs Locais com Ollama

Benchmark de modelos locais servidos pelo Ollama em Docker, sem chave de API e sem custo por token.

## Stack usado
- Servidor: `Ollama` em container Docker (`http://localhost:11434`)
- Modelos de chat: `qwen2.5:0.5b`, `qwen2.5:1.5b` e `llama3.2:1b`
- Modelo de embeddings: `nomic-embed-text`

Pré-requisito: subir o container com `docker compose up -d` e puxar os modelos (ver README do módulo).

In [ ]:
from pathlib import Path
import sys
import json

# Faz o notebook funcionar tanto na pasta do artigo quanto via raiz do repositório ou subpasta.
candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "0008_llms_locais_ollama",
    Path.cwd().parent,
    Path.cwd().parent / "0008_llms_locais_ollama",
]
ARTICLE_ROOT = next((p.resolve() for p in candidate_dirs if (p / "src" / "ollama_lab.py").exists()), Path.cwd())
if str(ARTICLE_ROOT) not in sys.path:
    sys.path.insert(0, str(ARTICLE_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.ollama_lab import (
    DEFAULT_BASE_URL,
    DEFAULT_CHAT_MODELS,
    DEFAULT_EMBEDDING_MODEL,
    check_server,
    run_lab,
)

## 1. Healthcheck do servidor local

Antes de medir qualquer coisa, provamos que o servidor existe, responde e tem os modelos baixados. Esse passo substitui o cadastro em plataforma de nuvem: quem garante o ambiente aqui é o Docker na sua máquina.

In [ ]:
server = check_server(DEFAULT_BASE_URL)
print(f"Ollama {server['version']} em {DEFAULT_BASE_URL}")
pd.DataFrame(server["models"])

## 2. Benchmark de chat: latência, TTFT e vazão

O runner envia os mesmos prompts para cada modelo e mede:

- `ttft_ms`: tempo até o primeiro token (a métrica que define a sensação de velocidade)
- `total_ms`: latência total da chamada
- `tokens_por_segundo`: vazão de geração no seu hardware

In [ ]:
resultados, resumo, estruturado, embeddings = run_lab(
    base_url=DEFAULT_BASE_URL,
    chat_models=DEFAULT_CHAT_MODELS.split(","),
    embedding_model=DEFAULT_EMBEDDING_MODEL,
    repeat=2,
    attempts=5,
)
resumo

## 3. Onde cada modelo acerta e erra

Nem toda diferença aparece na média. Vale inspecionar a resposta por prompt: modelos menores costumam emendar classificação com geração de texto quando deveriam responder apenas o rótulo.

In [ ]:
resultados[["modelo", "prompt_id", "ttft_ms", "total_ms", "tokens_gerados", "tokens_por_segundo", "acertou"]]

In [ ]:
plt.figure(figsize=(9, 4))
for modelo, grupo in resultados.groupby("modelo"):
    plt.scatter(grupo["total_ms"], grupo["tokens_por_segundo"], label=modelo, s=60)
plt.title("Latência x vazão dos modelos locais")
plt.xlabel("Latência total (ms)")
plt.ylabel("Tokens/s")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. Saída estruturada: o modelo sustenta um planner?

O agente do artigo 0007 depende de JSON válido. Aqui testamos a mesma exigência contra cada modelo local, com `format: "json"` forçado no Ollama.

In [ ]:
estruturado

## 5. Embeddings locais: retrieval sem nuvem

A mesma instância que serve o chat serve embeddings. O roteamento semântico do artigo 0007 pode, portanto, rodar inteiro no seu computador.

In [ ]:
rotas = pd.read_csv(ARTICLE_ROOT / "data" / "embedding_routes.csv")
embeddings

In [ ]:
rotas

## 6. Artefatos gerados

- `data/benchmark_resultados.csv` - linha a linha de cada chamada medida
- `data/benchmark_resumo.csv` - consolidação por modelo
- `data/structured_output.csv` - taxa de JSON válido e tool correta
- `data/embedding_routes.csv` - rotas semânticas resolvidas localmente
- `data/benchmark_relatorio.md` - relatório pronto para leitura
- `data/modelos_disponiveis.json` - snapshot do servidor no momento da execução

In [ ]:
print((ARTICLE_ROOT / "data" / "benchmark_relatorio.md").read_text(encoding="utf-8"))

## Próximos passos

1. Repita o benchmark com os modelos que você pretende usar em produção local.
2. Compare o TTFT com o SLA do seu produto antes de escolher o modelo.
3. Use o modelo vencedor como planner no agente do artigo 0007 e rode os evals do artigo 0006.